<a href="https://colab.research.google.com/github/rileykinlaw99-alt/authentication-project/blob/main/Authentication_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
import sqlite3
import hashlib
import os
from datetime import datetime

In [21]:
class SimpleAuthSystem:
    def __init__(self, db_name="security.db"):
        self.conn = sqlite3.connect(db_name)
        self.cursor = self.conn.cursor()
        self.setup_database()

    def setup_database(self):
        """Creates relational tables for user credentials and audit logs."""
        self.cursor.execute("""
            CREATE TABLE IF NOT EXISTS users (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                username TEXT UNIQUE NOT NULL,
                password_hash TEXT NOT NULL,
                salt TEXT NOT NULL
            )
        """)

        self.cursor.execute("""
            CREATE TABLE IF NOT EXISTS login_logs (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                timestamp TEXT NOT NULL,
                username TEXT NOT NULL,
                status TEXT NOT NULL
            )
        """)
        self.conn.commit()

    def _hash_password(self, password: str, salt: bytes = None) -> tuple[str, str]:
        """Hashes a password using PBKDF2-HMAC-SHA256 with a unique salt."""
        if not salt:
            salt = os.urandom(16)
        pwd_hash = hashlib.pbkdf2_hmac('sha256', password.encode('utf-8'), salt, 100000)
        return pwd_hash.hex(), salt.hex()

    def register_user(self, username: str, password: str):
        """Registers a new user using parameterized queries to prevent SQL Injection."""
        pwd_hash, salt = self._hash_password(password)
        try:
            self.cursor.execute(
                "INSERT INTO users (username, password_hash, salt) VALUES (?, ?, ?)",
                (username, pwd_hash, salt)
            )
            self.conn.commit()
            print(f"[SUCCESS] User '{username}' registered successfully.")
        except sqlite3.IntegrityError:
            print(f"[ERROR] Username '{username}' is already taken.")

    def login(self, username: str, password: str) -> bool:
        """Authenticates user credentials and logs the access attempt."""
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

        self.cursor.execute("SELECT password_hash, salt FROM users WHERE username = ?", (username,))
        result = self.cursor.fetchone()

        if result:
            stored_hash, stored_salt = result

            # Dynamically handle string or bytes to prevent TypeErrors
            if isinstance(stored_salt, str):
                salt_bytes = bytes.fromhex(stored_salt)
            else:
                salt_bytes = stored_salt

            input_hash, _ = self._hash_password(password, salt_bytes)

            if input_hash == stored_hash:
                self._log_attempt(timestamp, username, "SUCCESS")
                print(f"[LOGIN SUCCESS] Welcome back, {username}!")
                return True

        self._log_attempt(timestamp, username, "FAILED")
        print("[LOGIN FAILED] Invalid username or password.")
        return False

    def _log_attempt(self, timestamp: str, username: str, status: str):
        """Appends an access event to the database audit logs."""
        self.cursor.execute(
            "INSERT INTO login_logs (timestamp, username, status) VALUES (?, ?, ?)",
            (timestamp, username, status)
        )
        self.conn.commit()

    def view_audit_logs(self):
        """Fetches and displays recorded security audit events."""
        print("\n--- Security Audit Logs ---")
        self.cursor.execute("SELECT timestamp, username, status FROM login_logs")
        logs = self.cursor.fetchall()
        for log in logs:
            print(f"[{log[0]}] User: {log[1]} | Status: {log[2]}")

In [22]:
if __name__ == "__main__":
    # Re-instantiate the auth system on the fresh database
    auth = SimpleAuthSystem()

    # 1. Register test user
    auth.register_user("alex_dev", "SecurePassword123!")

    # 2. Test successful login
    auth.login("alex_dev", "SecurePassword123!")

    # 3. Test failed login
    auth.login("alex_dev", "WrongPassword!")

    # 4. View logs
    auth.view_audit_logs()

[ERROR] Username 'alex_dev' is already taken.
[LOGIN SUCCESS] Welcome back, alex_dev!
[LOGIN FAILED] Invalid username or password.

--- Security Audit Logs ---
[2026-10-06 10:07:50] User: alex_dev | Status: SUCCESS
[2026-10-06 10:07:50] User: alex_dev | Status: FAILED
[2026-10-06 10:08:44] User: alex_dev | Status: SUCCESS
[2026-10-06 10:08:44] User: alex_dev | Status: FAILED
[2026-10-06 10:10:34] User: alex_dev | Status: SUCCESS
[2026-10-06 10:10:34] User: alex_dev | Status: FAILED
[2026-10-06 10:11:06] User: alex_dev | Status: SUCCESS
[2026-10-06 10:11:07] User: alex_dev | Status: FAILED
[2026-10-06 10:11:13] User: alex_dev | Status: SUCCESS
[2026-10-06 10:11:14] User: alex_dev | Status: FAILED


In [23]:
# --- Additional Test ---
if __name__ == "__main__":
    # Test 1: Register a second user with special characters
    auth.register_user("kim_security", "P@ssw0rd_2026!#")

    # Test 2: Attempt login with a non-existent username
    print("\n[Test: Non-existent User]")
    auth.login("ghost_user", "SomePassword123!")

    # Test 3: Attempt login with correct username but wrong password
    print("\n[Test: Incorrect Password]")
    auth.login("kim_security", "WrongPassword!")

    # Test 4: Successful login for the new user
    print("\n[Test: Correct Credentials]")
    auth.login("kim_security", "P@ssw0rd_2026!#")

    # View updated audit logs to verify all attempts were recorded
    auth.view_audit_logs()

[SUCCESS] User 'kim_security' registered successfully.

[Test: Non-existent User]
[LOGIN FAILED] Invalid username or password.

[Test: Incorrect Password]
[LOGIN FAILED] Invalid username or password.

[Test: Correct Credentials]
[LOGIN SUCCESS] Welcome back, kim_security!

--- Security Audit Logs ---
[2026-10-06 10:07:50] User: alex_dev | Status: SUCCESS
[2026-10-06 10:07:50] User: alex_dev | Status: FAILED
[2026-10-06 10:08:44] User: alex_dev | Status: SUCCESS
[2026-10-06 10:08:44] User: alex_dev | Status: FAILED
[2026-10-06 10:10:34] User: alex_dev | Status: SUCCESS
[2026-10-06 10:10:34] User: alex_dev | Status: FAILED
[2026-10-06 10:11:06] User: alex_dev | Status: SUCCESS
[2026-10-06 10:11:07] User: alex_dev | Status: FAILED
[2026-10-06 10:11:13] User: alex_dev | Status: SUCCESS
[2026-10-06 10:11:14] User: alex_dev | Status: FAILED
[2026-10-06 10:12:53] User: ghost_user | Status: FAILED
[2026-10-06 10:12:53] User: kim_security | Status: FAILED
[2026-10-06 10:12:53] User: kim_securit